# Flow Matching on $\mathrm{SE}(2) \times \mathbb{R}^2$
In this notebook, we perform flow matching on the direct product of the roto-translation group $\mathrm{SE}(2)$ and the translation group $\mathbb{R}^2$, corresponding to Figure 3 in ["Flow Matching on Lie Groups"](https://arxiv.org/abs/2504.00494).

In [ ]:
import torch
import numpy as np
from lieflow.groups import (
    SE2,
    Rn,
    SE2byRn,
    _mod_offset
)
from lieflow.models import (
    get_model_FM,
    LogarithmicDistance,
    DataGenerator,
)
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from tqdm.notebook import tqdm
import json
with open("plot.json", "r") as f:
    plot_config = json.load(f)
latex_line_width = plot_config.pop("latex_line_width")
plt.rcParams.update(plot_config)

device = "cpu"

In [ ]:
se2 = SE2()
r2 = Rn(2)
se2byr2 = SE2byRn(se2, r2)

In [ ]:
generate_videos = True

# Distributions

In [ ]:
def generate_normals_SE2(N):
    translations = torch.randn(N, 2, device=device)
    rotations = 2 * torch.pi * torch.rand(N, device=device)
    g = torch.hstack((translations, rotations[..., None]))
    return g

def generate_uniforms_on_circle_SE2(N, centre=(0., 0.), ε=0.05):
    centre = torch.tensor(centre, device=device)
    normals = torch.randn(N, 2, device=device)
    lengths = (normals**2).sum(dim=-1).sqrt()
    non_zero = lengths > 0.
    circle_samples = normals[non_zero] / lengths[non_zero, None]
    angles = _mod_offset(torch.arctan2(circle_samples[..., 1], circle_samples[..., 0]), 2 * torch.pi, 0.)
    gs = torch.hstack((circle_samples + centre, angles[..., None]))
    return gs + ε * torch.randn(gs.shape, device=device)

def generate_line_SE2(N, d=0., w=1., horizontal=True, ε=0.05):
    if horizontal:
        xs = 2 * w * (torch.rand(N, device=device) - 0.5)
        ys = torch.ones(N, device=device) * d
        angles = torch.ones(N, device=device) * torch.pi / 2.
    else:
        xs = torch.ones(N, device=device) * d
        ys = 2 * w * (torch.rand(N, device=device) - 0.5)
        angles = torch.zeros(N, device=device)
    gs = torch.hstack((xs[..., None], ys[..., None], angles[..., None]))
    return gs + ε * torch.randn(gs.shape, device=device)

def generate_normals_R2(N):
    return torch.randn(N, 2, device=device)

def generate_uniforms_on_circle_R2(N, ε=0.04):
    normals = generate_normals_R2(N)
    lengths = (normals**2).sum(dim=-1).sqrt()
    non_zero = lengths > 0.
    circle_samples = normals[non_zero] / lengths[non_zero, None]
    return circle_samples + ε * torch.randn(circle_samples.shape, device=device)

@torch.no_grad()
def plot_g(ax, g, **kwargs):
    g = g.to("cpu")
    ax[0].quiver(g[..., 0], g[..., 1], torch.cos(g[..., 2]), torch.sin(g[..., 2]), width=0.01, **kwargs)
    ax[1].scatter(g[..., -2], g[..., -1], marker=".", **kwargs)

In [ ]:
tests = ("line_by_normal_to_circle_by_circle",)

In [ ]:
def data_generator(test, ε=0.04):
    match test:
        case "line_by_normal_to_line_by_circle":
            def generate_g_0(n):
                return torch.hstack((
                    generate_line_SE2(n, d=-2.5, w=1.5, horizontal=False, ε=ε),
                    generate_normals_R2(n)
                ))
            def generate_g_1(n):
                return torch.hstack((
                    generate_line_SE2(n, d=2.5, w=1.5, horizontal=False, ε=ε),
                    generate_uniforms_on_circle_R2(n, ε=ε)
                ))
        case "line_by_normal_to_circle_by_circle":
            def generate_g_0(n):
                return torch.hstack((
                    generate_line_SE2(n, d=-2.5, w=1.5, horizontal=False, ε=ε),
                    generate_normals_R2(n)
                ))
            def generate_g_1(n): 
                return torch.hstack((
                    generate_uniforms_on_circle_SE2(n, centre=(1.25, 0.), ε=ε),
                    generate_uniforms_on_circle_R2(n, ε=ε)
                ))
    return generate_g_0, generate_g_1

# Models

## Training

In [ ]:
with open("config.json", "r") as f:
    config_space = json.load(f)["SE2_by_R2"]
# Parameters shared within the space, which may be overridden per test case.
config_shared = {key: value for key, value in config_space.items() if not isinstance(value, dict)}
config = {test: config_shared | value for test, value in config_space.items() if isinstance(value, dict)}

In [ ]:
def train_model(generate_g_0, generate_g_1, config):
    train_loader = DataGenerator(generate_g_0, generate_g_1, config["batch_size"], config["batches_per_epoch"])

    model_FM = get_model_FM(se2byr2, H=config["width"], L=config["depth"], shared_weights=config["shared_weights"]).to(device)
    print("Number of parameters: ", model_FM.parameter_count)
    optimizer_FM = torch.optim.Adam(model_FM.parameters(), config["learning_rate"], weight_decay=config["weight_decay"])
    loss = LogarithmicDistance(torch.tensor(config["loss_weights"], device=device))

    losses_FM = np.zeros(config["epochs"])
    for i in tqdm(range(config["epochs"])):
        losses_FM[i] = model_FM.train_network(device, train_loader, optimizer_FM, loss, show_progress=False)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.plot(losses_FM)
    ax.set_title("Batch Loss Flow Matching")
    ax.set_xscale("log")
    ax.set_yscale("log")

    return model_FM

In [ ]:
models_FM = {}
for test in tests:
    print(test)
    generate_g_0, generate_g_1 = data_generator(test, ε=config[test]["epsilon"])

    g_0s = generate_g_0(32)
    g_1s = generate_g_1(32)

    fig, ax = plt.subplots(2, 2, figsize=(10, 10), dpi=600)
    for a in ax.flatten():
        a.set_xlim(-3, 3)
        a.set_ylim(-3, 3)
    plot_g(ax[:, 0], g_0s[:32], color="tab:blue")
    ax[0, 0].set_title(r"$\mathfrak{X}_0$")
    plot_g(ax[:, 1], g_1s[:32], color="tab:blue")
    ax[0, 1].set_title(r"$\mathfrak{X}_1$")

    models_FM[test] = train_model(generate_g_0, generate_g_1, config[test])

## Testing

In [ ]:
def create_animations(tests, models_FM, N_samples=2**5, N_steps=120):
    N_models = len(tests)

    # Forward flow
    g_0s_test = {}
    g_ts_test = {}
    for test in tests:
        generate_g_0, _ = data_generator(test, ε=config[test]["epsilon"])
        g_0s_test[test] = generate_g_0(N_samples).to(device)
        g_ts_test[test] = g_0s_test[test].detach().clone()
        
        models_FM[test].eval()

    t = 0
    Δt = 1. / N_steps
    metadata = {'title': 'Flow Matching SE(2) x R^2', 'artist': 'Matplotlib'}
    writer = FFMpegWriter(fps=30, metadata=metadata)

    fig, ax = plt.subplots(2, N_models, figsize=(5 * N_models, 2 * 5))

    with writer.saving(fig, "output/flow_matching_SE2_by_R2.mp4", dpi=150):
        for i, test in enumerate(tests):
            g_ts_test_plot = g_ts_test[test].to("cpu")
            ax[0].quiver(
                g_ts_test_plot[:, 0], g_ts_test_plot[:, 1],
                torch.cos(g_ts_test_plot[:, 2]), torch.sin(g_ts_test_plot[:, 2]),
                color="tab:blue"
            )
            ax[0].set_xlim(-3, 3)
            ax[0].set_ylim(-3, 3)
            ax[0].set_title(r"$\mathfrak{X}_0$")
            ax[1].scatter(
                g_ts_test_plot[:, -2], g_ts_test_plot[:, -1]
            )
            ax[1].set_xlim(-3, 3)
            ax[1].set_ylim(-3, 3)
        writer.grab_frame()
        for frame in tqdm(range(N_steps)):
            t = frame * Δt
            for i, test in enumerate(tests):
                with torch.no_grad():
                    g_ts_test[test] = models_FM[test].step(g_ts_test[test], torch.tensor([t], device=device), Δt)
                ax[0].clear()
                ax[1].clear()
                plot_g(ax, g_ts_test[test], color="tab:blue")
                ax[0].set_title(fr"$\mathfrak{{X}}_{{{t:.2f}}}$")
                ax[0].set_xlim(-3, 3)
                ax[0].set_ylim(-3, 3)
                ax[1].set_xlim(-3, 3)
                ax[1].set_ylim(-3, 3)
            writer.grab_frame()
        writer.grab_frame()

In [ ]:
if generate_videos:
    create_animations(tests, models_FM, N_samples=2**5)

In [ ]:
t = 0
N_steps = config_shared["N_steps"]
Δt = 1. / N_steps
N_show = 5
N_skip = int(N_steps / (N_show-1))

N_models = len(tests)
N_samples = 32

g_0s_test = {}
g_1s_test = {}
g_ts_test = {}
for test in tests:
    generate_g_0, generate_g_1 = data_generator(test, ε=config[test]["epsilon"])
    g_0s_test[test] = generate_g_0(N_samples).to(device)
    g_1s_test[test] = generate_g_1(N_samples).to(device)
    g_ts_test[test] = g_0s_test[test].detach().clone()
    
    models_FM[test].eval()

fig = plt.figure(figsize=(4.8, 2 * 1.6 * N_models * 3/3.1))
gs = gridspec.GridSpec(2 * N_models, 4, width_ratios=[1, 1, 1, 0.1], height_ratios=2 * N_models * [1.], wspace=0.1, hspace=0.1)
cax = fig.add_subplot(gs[:, 3])

ax = []
for i in range(2 * N_models):
    ax.append([])
    for j in range(3):
        a = fig.add_subplot(gs[i, j])
        a.set_xlim(-3, 3)
        a.set_ylim(-3, 3)
        a.set_xticks([])
        a.set_yticks([])
        a.set_aspect("equal")
        ax[i].append(a)
ax = np.array(ax)
        
ax[0, 0].set_title(r"$\mathfrak{X}_0$")
ax[0, 1].set_title(r"$\mathfrak{X}_t$")
ax[0, 2].set_title(r"$\mathfrak{X}_1$")
Δc = 1 / (N_show - 1)
colors = [(j * Δc, 0.1, 1 - j * Δc) for j in range(N_show)]
cmap = mcolors.ListedColormap(colors)
for i, test in enumerate(tests):
    k = 0

    plot_g(ax[2*i:2*i+2, 0], g_0s_test[test][:N_samples], color="tab:blue")
    plot_g(ax[2*i:2*i+2, 2], g_1s_test[test][:N_samples], color="tab:blue")

    alpha = 1
    N_samples_shown = N_samples
    for j in range(N_steps+1):
        t = j * Δt
        if j == N_steps:
            alpha = 1
            N_samples_shown = N_samples
        with torch.no_grad():
            g_ts_test[test] = models_FM[test].step(g_ts_test[test], torch.tensor([t], device=device), torch.tensor([Δt], device=device))

        if j % N_skip == 0:
            plot_g(ax[2*i:2*i+2, 1], g_ts_test[test], color=colors[k], alpha=alpha)
            k += 1
        alpha = 0.5
        N_samples_shown = N_samples // 4
fig.colorbar(ScalarMappable(cmap=cmap), cax=cax, ticks=np.linspace(0, 1, N_show), label="$t$");
fig.savefig("output/interpolation_SE2_by_R2.pdf", bbox_inches="tight")